Aca lo que hacemos es preparar los datos para la red neuronal, vamos a limpiar, convertir texto a números, dividir en entrenamiento y prueba, y escalar.

In [14]:
#Cargamos y limpiamos
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# TotalCharges: pasar de texto a número y rellenar los vacíos con 0 (clientes nuevos)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# customerID es solo un código, no sirve para predecir
df = df.drop(columns="customerID")

# Churn en número: 1 = se fue, 0 = se quedó
df["Churn"] = (df["Churn"] == "Yes").astype(int)

print(df.shape)

(7043, 20)


In [15]:
# vamos a unir "No internet service" y "No phone service"
df = df.replace({"No internet service": "No", "No phone service": "No"})

In [16]:
# Feature engineering: cuántos servicios extra tiene cada cliente
servicios = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
             "TechSupport", "StreamingTV", "StreamingMovies"]
df["num_servicios"] = (df[servicios] == "Yes").sum(axis=1)

In [17]:
#Vamos a convertir texto a numero (get_dummies)
df = pd.get_dummies(df, drop_first=True, dtype=int)

print(df.shape)
df.head()

(7043, 25)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,num_servicios,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,...,DeviceProtection_Yes,TechSupport_Yes,StreamingTV_Yes,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,1,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0
1,0,34,56.95,1889.50,0,2,1,0,0,1,...,1,0,0,0,1,0,0,0,0,1
2,0,2,53.85,108.15,1,2,1,0,0,1,...,0,0,0,0,0,0,1,0,0,1
3,0,45,42.30,1840.75,0,3,1,0,0,0,...,1,1,0,0,1,0,0,0,0,0
4,0,2,70.70,151.65,1,0,0,0,0,1,...,0,0,0,0,0,0,1,0,1,0


In [18]:
#Separar entradas X y objetivo Y
X = df.drop(columns="Churn")   # lo que el modelo usa para predecir
y = df["Churn"]                # lo que el modelo debe predecir

In [19]:
#Dividimos los datos para entrenar y probar
# 80% para entrenar y 20% para probar
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Churn en train:", y_train.mean())
print("Churn en test:", y_test.mean())

Train: (5634, 24)  Test: (1409, 24)
Churn en train: 0.2653532126375577
Churn en test: 0.2654364797728886


In [20]:
#Escalar con MinMaxScaler
scaler = MinMaxScaler()

# fit_transform aprende los mínimos y máximos SOLO con train
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)

# test solo se transforma con lo aprendido en train
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

X_train.describe().loc[["min", "max"]]

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,num_servicios,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_Yes,...,DeviceProtection_Yes,TechSupport_Yes,StreamingTV_Yes,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
max,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [21]:
#Guardamos los archivos
train = X_train.copy()
train["Churn"] = y_train.values
test = X_test.copy()
test["Churn"] = y_test.values

train.to_csv("../data/processed/train.csv", index=False)
test.to_csv("../data/processed/test.csv", index=False)

joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(list(X.columns), "../models/columnas.pkl")

['../models/columnas.pkl']